# Module 3 — Prompt Engineering for Network Ops

**NetOps Co. · CELL-031A · ~35 minutes**

Module 1 produced prose for a human to read. This notebook produces **JSON that code can act on** —
which is the bridge from prompting to agents.

Three techniques, then the thing that makes them reliable.

## Setup — about 60 seconds

Run this once per session. Colab gives you a fresh machine each time, so the clone and
install have to happen again — that is normal, not a mistake.

**Your API key.** Click the 🔑 key icon in the left sidebar, add a secret named
`GEMINI_API_KEY`, and toggle *Notebook access* on. Get a free key at
[aistudio.google.com](https://aistudio.google.com) — no credit card.

Never paste a key into a cell. Notebooks get shared, and the key goes with them.

In [ ]:
!git clone -q https://github.com/telcobytes/netops-genai-course.git 2>/dev/null || (cd netops-genai-course && git pull -q)
!pip install -q google-genai pydantic

import sys, os
sys.path.append('/content/netops-genai-course/data')

# Key from Colab secrets, with a local fallback so this notebook also runs
# in plain Jupyter.
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    if not os.environ.get('GEMINI_API_KEY'):
        import getpass
        os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')

print('Key loaded:', bool(os.environ.get('GEMINI_API_KEY')))
print('Module 3 — KPI Anomaly Explainer')

### Sanity check — no API key needed

`mock_tools.py` is pure standard library. If this prints numbers, your environment is
working and the rest of the notebook will run.

In [ ]:
import mock_tools
summary = mock_tools.get_cell_kpis('CELL-031A')
print('window     :', summary['window_start'], '->', summary['window_end'])
print('samples    :', summary['sample_count'])
print('rolling avg:', summary['rolling_avg'])
for c in summary['thresholds_crossed']:
    print(f"  CROSSED  {c['metric']} = {c['value']} ({c['comparison']} {c['threshold']})")

---
## 1. What the tool hands you

`get_cell_kpis` does not dump raw counters. It computes rolling averages, deltas and threshold
crossings, and hands the agent a small, already-correct summary.

**Tools compute. Agents reason.** Never make a language model do arithmetic over a long table —
it wastes context and invites confident arithmetic errors.

In [ ]:
import mock_tools, nb_viz
from IPython.display import HTML, display

k = mock_tools.get_cell_kpis('CELL-031A')
crossed = {c['metric'] for c in k['thresholds_crossed']}

display(HTML(nb_viz.table(
    k['readings'],
    columns=['timestamp','prb_utilization_pct','rrc_setup_success_rate_pct',
             'rrc_drop_rate_pct','active_users'],
    highlight=lambda row, col: col in crossed and float(row[col]) and (
        float(row[col]) > 75 if 'prb' in col else
        float(row[col]) > 5 if 'drop' in col else float(row[col]) < 95),
    caption='CELL-031A — red cells are past an operational threshold')))

print('rolling avg:', k['rolling_avg'])
print('delta      :', k['delta'])

---
## 2. Zero-shot vs few-shot — the categorization problem

Ask twice, loosely, and watch the category wording drift. That drift is fatal the moment code
has to branch on the answer.

In [ ]:
from llm_client import call_llm
import json

loose = f'Categorize this anomaly in a few words: {json.dumps(k["rolling_avg"])}'
for i in range(3):
    print(f'run {i+1}:', call_llm([{'role':'user','content':loose}]).strip()[:90])

In telecom, few-shot is not about tone — it is **domain taxonomy enforcement**. Give it the
buckets and two worked examples, and the drift stops.

In [ ]:
DOMAINS = ['RADIO_ACCESS_INTERFERENCE','CAPACITY_PRB_EXHAUSTION',
           'TRANSPORT_BACKHAUL_JITTER','CORE_SIGNALING_REJECT']

few_shot = f'''Categorize into EXACTLY ONE of: {', '.join(DOMAINS)}

Example — PRB 45%, backhaul RTT 120ms above baseline -> TRANSPORT_BACKHAUL_JITTER
Example — PRB 94%, users 210 vs capacity 150      -> CAPACITY_PRB_EXHAUSTION

READINGS: {json.dumps(k['rolling_avg'])}
Reply with the domain name only.'''

for i in range(3):
    print(f'run {i+1}:', call_llm([{'role':'user','content':few_shot}]).strip())

---
## 3. Structured output, enforced — with Pydantic

Asking nicely for JSON gets JSON *most of the time*. Most of the time is fine for prose and
useless for anything your code parses.

Two levels of enforcement, and you want both:

- `json_mode=True` makes the **API** return valid JSON
- a **Pydantic model** makes sure the JSON means what you expect — right fields, right types, right enum

This is the same mechanism that guards tool arguments in Module 10. Learn it here.

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal, List

class AnomalyReport(BaseModel):
    metrics_changed: List[str]
    thresholds_crossed: List[str]
    fault_category: Literal['RADIO_ACCESS_INTERFERENCE','CAPACITY_PRB_EXHAUSTION',
                            'TRANSPORT_BACKHAUL_JITTER','CORE_SIGNALING_REJECT']
    summary: str = Field(min_length=20, max_length=300)

PROMPT = '''Identify:
1. metrics_changed - which metrics moved most
2. thresholds_crossed - which operational thresholds were crossed
3. fault_category - ONE of: RADIO_ACCESS_INTERFERENCE, CAPACITY_PRB_EXHAUSTION,
   TRANSPORT_BACKHAUL_JITTER, CORE_SIGNALING_REJECT
4. summary - one plain-language sentence

Respond ONLY as JSON with those four keys.
READINGS: {readings}'''

raw = call_llm([{'role':'user','content': PROMPT.format(readings=json.dumps(k['rolling_avg']))}],
               json_mode=True)
report = AnomalyReport(**json.loads(raw))
report

Now watch it **reject** something wrong. This is the failure you want — loud, immediate, and free.

In [ ]:
try:
    AnomalyReport(metrics_changed=['prb'], thresholds_crossed=['prb'],
                  fault_category='SOMETHING_I_MADE_UP', summary='short')
except ValidationError as e:
    print(e)

> Two errors caught in microseconds: an invented category, and a summary below the minimum length.
> No API call, no cost, no chance of being wrong about it.

---
## Your turn

1. Run the explainer against **CELL-022A** and confirm it reports nothing wrong
2. Add a `confidence: float = Field(ge=0, le=1)` field to the model and update the prompt
3. Delete the few-shot examples from the prompt and see how often the category validates

In [ ]:
# Your turn
k2 = mock_tools.get_cell_kpis('CELL-022A')
print('thresholds crossed on CELL-022A:', k2['thresholds_crossed'] or 'none')

raw2 = call_llm([{'role':'user','content': PROMPT.format(readings=json.dumps(k2['rolling_avg']))}],
                json_mode=True)
print(AnomalyReport(**json.loads(raw2)))

---
**Next:** the explainer only knows the numbers you hand it. It has no idea NetOps Co. has seen
this pattern before. Module 4 teaches it to remember.